
<h1 id="Flat-field-images">Flat field images</h1><p>A <strong>flat field image</strong> or <strong>flat field frame</strong> is a calibration image taken of a uniform, evenly illuminated source to correct for optical system and camera sensor variations.
In a perfect telescope with an ideal instrument, you would expect to record on a given pixel the same number of photons entering through the <em>entrance pupil</em> of the telescope for the corresponding direction. If you illuminate <em>homogeneously</em> the telescope pupil, you would expect each pixel to record the same number of photons. In reality, despite your best attempts to homogeneously illuminate the telescope pupil, each pixel will record a slightly different value from the surrounding one, and even larger differences may be apparent when considering the entire CCD.</p>
<p>The most significant sources of variation are:</p>
<ul>
<li><strong>dust or dishomogeneities along the optical path</strong>: specs of dust on the mirror or on the sensor, and in general any dishomogeneity along the optical path of the telescope and instrument combined, including the filter used to take the scientific images, can cause small dark spots or smudges;</li>
<li><strong>pixel sensitivity variations</strong>: each pixel of  a sensor can have slightly different sensitivity, which needs to be equalised for accurate measurements;</li>
<li><strong>cosmetic effects</strong>: some pixels may be damaged or affected by production problems, thus systematically recording a lower/higher number than photons. In some cases, the production technique (e.g., stamping a 512x512 pixel matrix on the silicon) may produce defects over hundreds of pixels, all over the detector;</li>
<li><strong>non-uniform illumination (vignetting)</strong>: a darkening of the image corners that occurs due to the camera lens or telescope optics blocking some of the light.</li>
</ul>
<p>Flat fielding corrects the combined optical system and CCD throughput at each pixel so that each pixel on the CCD responds equally to a source with the same photon flux. Flat fielding removes the effects of pixel-to-pixel sensitivity variations across the sensor, dust or scratches on the CCD window, and vignetting by the telescope or camera lens optics.  <br/>
We must use a flat-field frame for each filter/lens/telescope combination to obtain good results, e.g., different filters will have different production issues. The flat-field frames are obtained by photographing an evenly illuminated target at very close range, and therefore out of focus (<strong>dome flats</strong>), or by shooting the sky near dusk or dawn, when the sky is dim enough to avoid saturation of the sensor but still considerably brighter than the stars in the field (<strong>sky flats</strong>). The former technique usually involves a flat white target illuminated by a halogen lamp, and it is preferable because the light level is more easily controlled and can be carefully controlled at leisure. The latter techniques must be executed over a short time interval, possibly on a pre-selected field of view with a reduced number of stars, and, of course, with good weather conditions. If the CCD is cooled and the exposures are reasonably short, we can neglect the dark (thermal) count.</p>
<p>A flat field frame is characterised by <em>all</em> the sources of noise introduced in the previous lectures (read-out noise, dark current, photon noise, pixel response non-uniformity), with only the dark current being negligible, as the exposure time for a flat frame is usually a few seconds. <br/>
It is essential to take multiple exposures to reduce all sources of noise in the final <em>flat field calibration image</em>. For each frame, we should aim to reach the maximum number of counts to minimise the impact of photon noise; however, we must be careful to avoid <em>saturation</em> and the so-called <strong>non-linear regime</strong>, where the number of recorded photoelectrons no longer increases linearly with exposure time.</p>



<h1 id="Flat-analysis">Flat analysis</h1>



<p>We load the standard packages for data analysis and plotting <br/>
<a href="https://numpy.org/">NumPy</a> <br/>
<a href="https://matplotlib.org/">Matplotlib</a> <br/>
<a href="https://docs.astropy.org/en/stable/index.html">AstroPy</a> <br/>
<a href="https://docs.python.org/3/library/pickle.html">pickle</a></p>


In [ ]:


import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import pickle

%matplotlib widget 





<p>We read the list of bias frames in the usual way:</p>


In [ ]:


flat_list = np.genfromtxt('./group00_GJ3470_20220328/flat/flat.list', dtype=str)
print(flat_list)





<p>We also need to open the median bias and copy the estimates for the error associated with the bias and the readout noise obtained when analyzing the bias frames.
For this example, I will use the following values from the <code>lecture01\_bias\_analysis</code> Jupyter Notebook:</p>
<ul>
<li><code>bias_std</code>: Median STD of each pixel</li>
<li><code>readout_noise</code>: STD single frame</li>
<li><code>gain</code>: as extracted from the header of a bias frame</li>
</ul>


In [ ]:


median_bias = pickle.load(open("./group00_GJ3470_20220328/median_bias.p", "rb"))
bias_std = 1.3 # [e] photoelectrons
readout_noise = 7.4  # [e] photoelectrons
gain = 1.91 # [e/ADU]





<p>For illustration purposes, I am opening the first flat frame and extracting the data. <br/>
As a safety check, I'm also printing a few values from the header, directly from the first HDU. <br/>
Notice how I condensed all the steps that I had to take to go from the fits file to the header and header comments in one single call.</p>


In [ ]:


flat00_fits = fits.open('./group00_GJ3470_20220328/flat/'+flat_list[0])
flat00_data = flat00_fits[0].data * gain

print('CCD Gain         : {0:4.2f} {1:.8s}'.format(flat00_fits[0].header['GAIN'],flat00_fits[0].header.comments['GAIN']))
print('CCD Readout noise: {0:4.2f} {1:.3s}'.format(flat00_fits[0].header['RDNOISE'],flat00_fits[0].header.comments['RDNOISE']))
print('Shape of the FITS image from the header : {0:4d} x {1:4d} pixels'.format(flat00_fits[0].header['NAXIS1'], flat00_fits[0].header['NAXIS2']))





<h2 id="Overscan">Overscan</h2><p>It's always a good idea to visually inspect your data.
If we let <code>matplotlib</code> automatically choose the range of the colorbar, and we plot the average distribution of counts as a function of the column number, we can see that there are two strips on the side of the image with a much lower number of counts. Each column takes the name of <strong>overscan</strong>.</p>


In [ ]:


fig, ax = plt.subplots(2,1, figsize=(8,6)) # Caution, figsize will also influence positions.
im1 = ax[0].imshow(flat00_data, origin='lower')
median_column = np.average(flat00_data, axis=0)
im2 = ax[1].plot(median_column)

# add the colorbar using the figure's method,
# telling which mappable we're talking about and
# which axes object it should be near
cbar = fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("e")

ax[0].set_xlabel('X [pixels]')
ax[0].set_ylabel('Y [pixels]')

ax[1].set_xlabel('X [pixels]')
ax[1].set_ylabel('Y [pixels]')
plt.show()





<p>The overscan region of a CCD is a part of the sensor that is covered so that it never receives any light, thus acting like a <em>local</em> bias. Depending on the sensor, it can be a valuable way to remove small variations in the bias level from frame to frame. Alternatively, you can use it to remove the bias level when the bias frames are not available.</p>
<p>Generally speaking, you can replace the median bias with the overscan only if there is no variation along the horizontal axis. Otherwise, it is preferable to use the median bias, if available.</p>



<h2 id="Properly-dealing-with-the-entire-frame">Properly dealing with the entire frame</h2><p>The overscan regions do not impact our data reduction and science analysis, but their presence is annoying when performing statistics on the full frame or displaying it. There are (at least) two ways to deal with this problem:</p>
<ol>
<li>We can trim the outer columns from all images and save the trimmed frames as new frames.</li>
<li>we can exclude the outer columns from the analysis when computing full-frame statistics and visualization.</li>
</ol>
<p>The first approach is well-diffused, but you need software that propagates the metadata (e.g., header, comments) into the new files. You also have to be sure to apply the same overscan cuts every time you reduce a new frame. You may also need extra space to store all the new images.
The second approach requires just a bit of attention.</p>
<p>Let's plot our frame again, but first, we compute the minimum and maximum values of our counts, excluding the overscan regions.</p>


In [ ]:


vmin = np.amin(flat00_data[:, 12:-12])
vmax = np.amax(flat00_data[:, 12:-12])
print(vmin, vmax)





<p>We do the plot again, but this time:</p>
<ol>
<li>we set the limits for the colorbar in <code>ax[0].imshow()</code></li>
<li>we set the limits for the vertical axis of the lower plot using <code>ax[1].set_ylim()</code></li>
</ol>


In [ ]:


fig, ax = plt.subplots(2,1, figsize=(8,6)) # Caution, figsize will also influence positions.
im1 = ax[0].imshow(flat00_data, origin='lower', vmin=vmin, vmax=vmax)
median_column = np.average(flat00_data, axis=0)
im2 = ax[1].plot(median_column)

# we set the plot limits 
ax[1].set_ylim(vmin,vmax)

# add the colorbar using the figure's method,
# telling which mappable we're talking about and
# which axes object it should be near
cbar = fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("e")

ax[0].set_xlabel('X [pixels]')
ax[0].set_ylabel('Y [pixels]')

ax[1].set_xlabel('X [pixels]')
ax[1].set_ylabel('Average counts [e]')
plt.show()





<h2 id="Computing-the-normalization-factors">Computing the normalization factors</h2><p>Computation of the median flat follows a similar strategy adopted for the median bias:</p>
<ol>
<li>we initialize a 3D array called <code>stack</code>, with dimensions equal to the number of images and the two dimensions of a frame</li>
<li>we read the frames one by one, <em>correct for bias</em>, and store them in the <code>stack.</code></li>
</ol>


In [ ]:


n_images = len(flat_list)
flat00_nparray_dim00, flat00_nparray_dim01 = np.shape(flat00_data)

stack = np.empty([n_images, flat00_nparray_dim00, flat00_nparray_dim01])

for i_flat, flat_name in enumerate(flat_list):
    flat_temp = fits.open('./group00_GJ3470_20220328/flat/'+flat_name)
    stack[i_flat,:,:] = flat_temp[0].data * flat_temp[0].header['GAIN'] - median_bias
    flat_temp.close()





<p>Before continuing, we have to remember the goal of flat calibration.</p>
<p>The purpose of flat corrections is to compensate for any non-uniformity in the response of the CCD to light. There can be several reasons for the non-uniform response across the detector:</p>
<ul>
<li>variations in the sensitivity of pixels in the detector.</li>
<li>dust on either the filter or the glass window covering the detector.</li>
<li>vignetting, a dimming in the corners of the image.</li>
<li>anything else in the optical path that affects how much light reaches the sensor.</li>
</ul>
<p>The fix for the non-uniformity is the same in all cases: take an image with uniform illumination and use that to measure the CCD's response.</p>
<p>When illuminating the CCD, we want to reach the highest signal-to-noise ratio for every pixel. As the photon noise goes with the square root of the flux, we need to achieve very high counts (without reaching saturation). However, dividing the science frames by these very high counts would produce unrealistically small photoelectron fluxes. Since we are interested in the relative response of the pixels, i.e., how the pixels behave with respect to the others, and not to the absolute efficiency of each pixel, we can express the flat correction as the correction value relative to the median response over a selected sample of pixels.
e CCD.</p>



<p>We can compute the reference value of each frame by taking the median within a box of 50x50 pixels in the centre of each frame. Keep in mind that this is only one of the many possible approaches.</p>
<p><strong>Note</strong>: we convert the <code>real</code> number into <code>integers</code> using the function <a href="https://numpy.org/doc/stable/reference/arrays.scalars.html#numpy.int16">numpy.int16</a> Alternatively, we can use the Python built-in function <a href="https://docs.python.org/3/library/functions.html#int">int</a>.</p>


In [ ]:


windows_size = 50
# x0, x1, y0, y1 represents the coordinates of the four corners
x0 = np.int16(flat00_nparray_dim01/2 - windows_size/2)
x1 = np.int16(flat00_nparray_dim01/2 + windows_size/2)
y0 = np.int16(flat00_nparray_dim00/2 - windows_size/2)
y1 = np.int16(flat00_nparray_dim00/2 + windows_size/2)

print('Coordinates of the box: x0:{0}, x1:{1}, y0:{2}, y1:{3}'.format(x0, x1, y0, y1))





<p>To compute the normalization factor of each frame, we can rely again on the <code>axis</code> argument of the function <a href="https://numpy.org/doc/stable/reference/generated/numpy.median.html"><code>numpy.median</code></a>.</p>
<p>In the case of the bias frame, we wanted to compute the median across the 30 bias frames for each pixel of the sensor, so we set <code>axis=0</code>, and the output was the same size as a frame. Here, we want to compute the median of a given set of pixels for each frame, and the resulting array should be the same size as the number of frames.</p>
<p>To accomplish this, it is sufficient to specify the two axes we want to include in computing the median, using round parenthesis as in the example below. The median will be calculated using all the values in the range <code>[y0:y1, x0:x1]</code> of each frame</p>
<p>The data among round parentheses is called <em>tuple</em>, and it is one of Python's four main data types. Tuples have the following properties:</p>
<ul>
<li>a tuple is a collection which is ordered and unchangeable;</li>
<li>tuples s are written with round bracket;</li>
<li>elements can be duplicated, and of different types</li>
</ul>
<p>You can read more about tuples <a href="https://www.w3schools.com/python/python_tuples.asp">here</a></p>


In [ ]:


normalization_factors = np.median(stack[:,y0:y1, x0:x1], axis=(1,2))
print('Number of normalization factors (must be the same as the number of frames): {0}'.format(np.shape(normalization_factors)))
print(normalization_factors)





<p>We can estimate the error of our normalisation factor by computing the standard deviation within the box (which represents the error associated with each pixel) and dividing by the square root of the number of points in the box. <br/>
<strong>This is not the correct way to propagate the errors</strong>, but it gives us a good estimate of the order of magnitude of the error associated with the normalisation factor.</p>


In [ ]:


normalization_factors_std = np.std(stack[:,y0:y1, x0:x1], axis=(1,2)) / np.sqrt(windows_size**2)
print(normalization_factors_std)




In [ ]:


plt.figure()
x_frame = np.arange(0,n_images, 1)
plt.scatter(x_frame, normalization_factors)
plt.errorbar(x_frame, normalization_factors, normalization_factors_std, fmt='o', ms=2)
plt.xlabel('Frame number')
plt.ylabel('Average counts [e]')
plt.show()





<p>From the plot above, we can see that the normalization factor is not constant from one image to another, although we would expect to measure the same flux for each pixel, as we are observing a fixed lamp for a predetermined exposure time. These oscillations are caused by a lamp problem discovered only some time after its installation. Depending on when your data has been gathered, you may not see this feature.</p>
<p>This is the main reason why we perform the normalization <em>before</em> computing the median flat. If the lamp's flux was perfectly constant across the frames, the order of normalization and median computation would not have mattered.
Note that we have to perform the normalization anyway, as we are interested in computing the sensor relative efficiency.</p>



<h2 id="Flat-normalization">Flat normalization</h2><p>To normalize each frame by its corresponding normlization factor, we can proceed in two ways:</p>
<ol>
<li>iterate along the frame, and perform the operation:</li>
</ol>


In [ ]:


stack_normalized_iter =  stack * 0. #initialization of the output array
for i_flat in range(n_images):
    stack_normalized_iter[i_flat,:,:] = stack[i_flat,:,:] / normalization_factors[i_flat]





<ol start="2">
<li>use the <a href="https://numpy.org/doc/stable/user/basics.broadcasting.html">broadcasting feature</a>.</li>
</ol>
<p>Broadcasting allows arithmetic operations between arrays with different shapes. Subject to certain constraints, the smaller array is “broadcast” across the larger array so that they have compatible shapes.</p>
<p>Broadcasting starts with the trailing (i.e., rightmost) dimension and works its way left. As this is the opposite situation of our case (<code>normalization_factors</code> has the same dimension as the leftmost dimension of <code>stack</code>), we can fix this nuisance by operating on the <em>transpose</em> of the <code>stack</code> array and then taking the transpose of the out to revert back to the original shape.
The transpose is computed by simply adding <code>.T</code> at the end of a Numpy array</p>


In [ ]:


print("shape of stack array           : ", np.shape(stack))
print("shape of transposed stack array: ", np.shape(stack.T))

stack_normalized = (stack.T / normalization_factors).T
## First alternative:  stack_normalized = np.divide(stack.T, normalization_factors).T 
## Second alternative: stack_normalized = np.multiply(stack.T, 1./normalization_factors).T 

print("shape of normalized stack array: ", np.shape(stack_normalized))





<p>We can verify if the two methods deliver the same results by checking the maximum deviations between the two arrays in absolute terms.</p>


In [ ]:


print("Maximum absolute difference between the two arrays: {0:2.6e}".format(np.max(np.abs(stack_normalized_iter-stack_normalized))))





<p>The value may not be zero due to the computer's precision:
<em>Epsilon</em> describes the round-off error for a floating-point number with a certain amount of precision. It can be thought of as the smallest number that can be added to 1.0 without changing its bits.</p>


In [ ]:


print(np.finfo(float).eps)





<p>Therefore, we can say that the two procedures produce identical results.</p>
<h2 id="Median-flat">Median flat</h2><p>We can finally compute the median normalized flat frame and store it on the computer:</p>


In [ ]:


median_normalized_flat = np.median(stack_normalized, axis=0)

pickle.dump(median_normalized_flat, open("median_normalized_flat.p", "wb")) 
pickle.dump(stack_normalized, open("flat_normalized_stack.p", "wb")) 
pickle.dump(normalization_factors, open("flat_normalization_factors.p", "wb")) 
pickle.dump(stack, open("flat_stack.p", "wb")) 




In [ ]:


i_image = 0
nmin = np.amin(median_normalized_flat[:, 12:-12])
nmax = np.amax(median_normalized_flat[:, 12:-12])
print(vmin, vmax)


fig, ax = plt.subplots(2,1, figsize=(8,6)) # Caution, figsize will also influence positions.
im1 = ax[0].imshow(median_normalized_flat, origin='lower', vmin=nmin, vmax=nmax)
median_column = np.average(median_normalized_flat, axis=0)
im2 = ax[1].plot(median_column)

# we set the plot limits 
ax[1].set_ylim(nmin,nmax)

# add the colorbar using the figure's method,
# telling which mappable we're talking about and
# which axes object it should be near
cbar = fig.colorbar(im1, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("e")

ax[0].set_xlabel('X [pixels]')
ax[0].set_ylabel('Y [pixels]')

ax[1].set_xlabel('X [pixels]')
ax[1].set_ylabel('Average value [normalized]')
plt.show()





<h1 id="Error-propagation">Error propagation</h1>



<p>Regarding the bias frames, the readout noise was the only contributor to the noise budget.
Now, we have three contributors:</p>
<ol>
<li>the readout noise</li>
<li>the error associated with the median bias frame</li>
<li>the photon noise associated with the flux of the lamp</li>
</ol>
<p>Dark current would contribute as well, but it can be neglected for short exposures and for modern detectors in general.</p>
<p>For 1) and 2), we already have an estimate of the error. For 3), we can assume that during the temporal interval covered by our exposure, photons reach the detector with a constant mean rate and independently of the time since the last event. (the arrival of one photon is not influenced by the other photons). In other words, photos follow a <a href="https://en.wikipedia.org/wiki/Poisson_distribution">Poisson distribution</a>; as such, the <em>variance</em> (i.e., the expected value of the squared deviation from the mean of a random variable.) is equal to the number of events, specifically the number of photoelectrons recorded.</p>
<p>Noting that we have already removed the bias value (which does not contribute to the photon noise), the photon noise associated with each measurement of each frame is given by the square root of the stack.</p>


In [ ]:


#photon_noise = np.sqrt(stack)
photon_noise = np.sqrt(np.abs(stack))





<p><strong>Note</strong> You may get a Runtime Error from the negative values in the overscan. You can either ignore it, or compute the square root of the absolute value of the stack</p>



<p>The total error associated with each frame <em>before</em> normalization is the sum in quadrature of the individual errors, as only sums and differences are involved in these data reduction steps.</p>


In [ ]:


stack_error = np.sqrt(readout_noise **2 + bias_std**2 +  photon_noise**2)





<p>Normalization is simply the division by a constant that we assume is errorless. According to the error propagation formula, we need to divide the error estimates by the same values.</p>
<p><strong>Optional task</strong> Compute the error associated with each normalization factor correctly and propagate its uncertainty to the median bias. You can use the definition of average to circumvent the fact that there is no analytical formulation for the median.</p>


In [ ]:


stack_normalized_error = (stack_error.T/normalization_factors).T 





<p>Finally, the error associated to the median flat can be computed by assuming that the <em>median function</em> (which cannot be expressed analytically) can be approximated by the <em>mean function</em>, which instead has an analytical expression and thus can be use with the error propagation formula.</p>
<p><strong>Note</strong> This is a strong assumption that we are making here specifically for our study case, in general you should always check if you are working in a regime of white noise. \</p>



<p>$ f = \sum\limits_{i=1}^{N} {f_i} / N $</p>



<p>$ \sigma_{f} = \left( \sqrt{ \sum\limits_{i=1}^{N} {\sigma^2_i}} \right) / N $</p>



<p>(demonstration is left to the reader)</p>


In [ ]:


## UPDATE on 03/11/2025: the square root of the sum was missing
median_normalized_flat_errors = np.sqrt(np.sum(stack_normalized_error**2, axis=0)) / n_images
print("shape of the median normalized error array: ", np.shape(median_normalized_flat_errors))
pickle.dump(median_normalized_flat_errors, open("median_normalized_flat_errors.p", "wb")) 





<h2 id="Some-statistics-on-the-flat">Some statistics on the flat</h2>



<p>One question should arise: why normalize first, and then computing the median?
Can I change the order without consequences?</p>
<p>First, let's see what is the distribution of the counts before and after normalization.
For this example, I'm considering pixels in the range [100:105,250:255]  (Python notation). The normalized values have been rescaled to the average of the normalization factors to allow a direct comparison.</p>
<p>If the flux of the lamp is constant, the two distributions should be extremely similar - almost identical.</p>


In [ ]:


mean_normalization = np.mean(normalization_factors)

plt.figure()
plt.hist(stack[:,40:45,250:255].flatten(), bins=20, alpha=0.5, label='Before norm.')
plt.hist(stack_normalized[:,40:45,250:255].flatten()*mean_normalization, bins=20, alpha=0.5, label='After norm.')
plt.xlabel('Counts [e]')
plt.ylabel('#')
plt.legend()
plt.show()





<p>There is a clear difference between the two distributions (can you quanitfy it)?
The reason appears clear if we plot the distribution of the normalization factors, and we compare it with the theoretical distribution that they should have if the variations was due to photon noise alone.</p>
<p>To compute the normal distribution, I assumed $\sigma = \sqrt{(f_{avg})}$</p>


In [ ]:


sigma_mean_normalization = np.sqrt(mean_normalization)
x = np.arange(np.amin(normalization_factors), np.amax(normalization_factors), 10)
y = 1./(sigma_mean_normalization * np.sqrt(2 * np.pi)) * \
               np.exp( - (x - mean_normalization)**2 / (2 * sigma_mean_normalization**2) )

plt.figure()
plt.hist(normalization_factors,alpha=0.5, density=True, label='Normalization factors')
plt.plot(x,y)
plt.xlabel('Counts [e]')
plt.ylabel('Probability density')
plt.legend()
plt.show()





<p>It appears clear that the variation in the illumination by the flat lamp is not consistent with photon noise alone. After technical investigation, a fluctuations in the voltage of the lamp has been discovered.</p>
<p>For this reason, we performed the normalization, i.e., removing the variation of illumination with time, before computing the median.</p>
